In [0]:
#reading the df using delta
df = spark.read.format("delta") \
    .table("transjakarta_dataset.silver.transjakarta_raw")
display(df)

In [0]:
#Filtering the flagged data and non flag data, and immediately write the flagged data to gold table
from pyspark.sql import functions as F
clean_df = df.filter(F.col("Note") == "tripSuccessful")
flagged_df = df.filter(F.col("Note") != "tripSuccessful")

#adding some gold table metadata
flagged_df = flagged_df.withColumn(
    "gold_processed_at", F.current_timestamp()
)

#writing the flag table to gold table
flagged_df.write  \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema", "true") \
.saveAsTable("transjakarta_dataset.gold.transjakarta_flagged")

print(f"Silver Total : {df.count()}")
print(f"clean Total : {clean_df.count()}")
print(f"flagged Total : {flagged_df.count()}")

In [0]:
#Adding journey_status column as label for better analysis later
clean_df = clean_df.withColumn("journey_status",F.lit("Compelete"))

In [0]:
clean_df = clean_df.withColumn(
    "journey_duration_seconds",
    (F.col("tapOutTime") - F.col("tapInTime")).cast("long")
).withColumn(
    "journey_duration_minutes",
    F.round(F.col("journey_duration_seconds") / 60,2)
).withColumn(
    "duration_validity",
     F.when(F.col("journey_duration_seconds") <= 0, F.lit("INVALID_DURATION"))
      .otherwise(F.lit("VALID"))
)

display(clean_df)

In [0]:
#making total revenue per day table
revenue_daily_df = clean_df.groupBy(F.to_date(F.col("tapInTime")).alias("date")).agg(F.sum(F.col("payAmount")).alias("total_revenue")).orderBy("date")
display(revenue_daily_df) 

In [0]:
#making busiest_route_df to see the busiest route per day
route_traffic_df = clean_df.groupBy(
    F.to_date(F.col("tapInTime")).alias("date"),
    F.dayofweek(F.col("tapInTime")).alias("day_of_week"),
    F.col("corridorID"),
    F.col("corridorName")
).agg(
    F.count("*").alias("total_journeys"),
    F.sum("payAmount").alias("total_revenue"),
    F.round(F.avg("journey_duration_minutes"), 2).alias("avg_duration_minutes")
).orderBy("date", F.desc("total_journeys"))
display(route_traffic_df)

In [0]:
#making day_traffic_df to see the busiest day of the week
day_traffic_df = clean_df.withColumn(
    "day_of_week",
    F.dayofweek(F.col("tapInTime"))
).withColumn(
    "is_weekend",
    F.when(F.col("day_of_week").isin(1, 7),F.lit("Weekend"))
     .otherwise(F.lit("Weekday"))
).groupBy("day_of_week", "is_weekend").agg(
    F.count("*").alias("total_journeys"),
    F.round(F.avg("payAmount"), 2).alias("avg_fare")
).orderBy(F.desc("total_journeys"))

display(day_traffic_df)

In [0]:
#making station_traffic_df to see the busiest stations
station_traffic_df = clean_df.groupBy(
    F.col("tapInStops").alias("station_id"),
    F.col("tapInStopsName").alias("station_name")
).agg(
    F.count("*").alias("total_tap_in")
).unionByName(
    clean_df.groupBy(
        F.col("tapOutStops").alias("station_id"),
        F.col("tapOutStopsName").alias("station_name")
    ).agg(
        F.count("*").alias("total_tap_in")
    )
).groupBy("station_id", "station_name").agg(
    F.sum("total_tap_in").alias("total_traffic")
).orderBy(F.desc("total_traffic"))

display(station_traffic_df)

In [0]:
# Write all table to gold

clean_df.write \
    .format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema","true")\
    .saveAsTable("transjakarta_dataset.gold.clean_transjakarta_dataset")

revenue_daily_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("transjakarta_dataset.gold.revenue_daily")

route_traffic_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("transjakarta_dataset.gold.route_traffic")

day_traffic_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("transjakarta_dataset.gold.traffic_by_day")

station_traffic_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("transjakarta_dataset.gold.station_footfall")

In [0]:
#Verifying every table already been sent to the gold medallion
spark.sql("SHOW TABLES IN transjakarta_dataset.gold").show()